In [ ]:




import cv2
from onnx2torch import convert
import torch
from insightface.app import FaceAnalysis
import numpy as np

app = FaceAnalysis(name='buffalo_l', providers=['CPUExecutionProvider'], allowed_modules=["detection", "genderage"])
app.prepare(ctx_id=0)



from abc import ABC, abstractmethod
from typing import List, Tuple
import numpy as np

class FaceModelBase(ABC):
    """
    Abstract base class for face-related models.
    Enforces common interface across detection and attribute models.
    """

    @abstractmethod
    def prepare(self, *args, **kwargs):
        """Optional setup before prediction (e.g., thresholds, sizes)."""
        pass

    @abstractmethod
    def predict(self, img: np.ndarray, bboxes: List[Tuple[float, float, float, float]]) -> List:
        """
        Predict outputs based on image and (optional) bounding boxes.
        - For detection models: `bboxes` is ignored, return new bboxes.
        - For attribute models: `bboxes` are used to crop/align faces.
        """
        pass






import cv2
import os
import numpy as np
import onnx
import torch
from onnx2torch import convert
from typing import Tuple, List
from insightface.utils.face_align import transform

class RetinaFaceTorch(FaceModelBase):
    """
    PyTorch-based port of the ONNXRuntime+NumPy RetinaFace wrapper.
    Usage:
        rft = RetinaFaceTorch(model_file="retinaface.onnx")
        rft.prepare(nms_thresh=0.4, det_thresh=0.5, input_size=(640,640))
        dets, kps = rft.detect(image)  # image as HxWxBGR uint8 numpy array
    """
    def __init__(self, model_file=None, input_size: Tuple[int, int] = (640, 640)):
        assert model_file is not None, "Must provide ONNX model file path"
        assert os.path.exists(model_file), f"Model file not found: {model_file}"
        # Convert ONNX to PyTorch
        onnx_model = onnx.load(model_file)
        self.torch_model = convert(onnx_model)
        self.torch_model.eval()
        # Set defaults
        self.model_file = model_file
        self.center_cache = {}
        self.nms_thresh = 0.4
        self.det_thresh = 0.5
        self.input_size = input_size
        self.input_mean = 127.5
        self.input_std = 128.0
        self.use_kps = False
        # Derive FPN settings by inspecting ONNX output count
        ort_sess = __import__('onnxruntime').InferenceSession(model_file, None)
        out_count = len(ort_sess.get_outputs())
        if out_count in (6, 9):
            self.fmc = 3
            self._feat_stride_fpn = [8, 16, 32]
        else:
            self.fmc = 5
            self._feat_stride_fpn = [8, 16, 32, 64, 128]
        # key 8/16/32/... anchors per stride
        # assume square anchor ratio 1.0, single anchor
        self._num_anchors = 1
        if out_count in (6, 9):
            self._num_anchors = 2
        if out_count in (9, 15):
            self.use_kps = True

    def prepare(self, nms_thresh=None, det_thresh=None, input_size=None):
        """Set thresholds and input canvas size (W,H)."""
        if nms_thresh is not None:
            self.nms_thresh = nms_thresh
        if det_thresh is not None:
            self.det_thresh = det_thresh
        if input_size is not None:
            self.input_size = input_size

    def forward(self, img):
        """
        Runs the PyTorch model on the preprocessed blob.
        Returns list of torch.Tensor outputs in the same order as ONNX.
        """
        H, W = img.shape[:2]
        # BlobFromImage: scale, mean/std normalization, swapRB
        blob = cv2.dnn.blobFromImage(
            img, 1.0/self.input_std, (W, H),
            (self.input_mean, self.input_mean, self.input_mean),
            swapRB=True
        )
        inp = torch.from_numpy(blob).float()
        with torch.no_grad():
            outs = self.torch_model(inp)
        # outs may be single tensor or list/tuple
        if isinstance(outs, torch.Tensor):
            outs = (outs,)
        return outs

    def detect(self, img, max_num=0, metric='default'):
        """
        Full detect pipeline: preprocess, forward, postprocess to bboxes/kps.
        img: HxWxBGR uint8
        returns: dets (N×5 numpy array: x1,y1,x2,y2,score), kps (N×5×2) or None
        """
        assert img.dtype == np.uint8, "Input must be uint8 BGR image"

        # Resize into input_size canvas
        if self.input_size is None:
            H, W = img.shape[:2]
            in_w, in_h = W, H
        else:
            in_w, in_h = self.input_size
        im_ratio = img.shape[0] / img.shape[1]
        model_ratio = in_h / in_w
        if im_ratio > model_ratio:
            new_h = in_h
            new_w = int(new_h / im_ratio)
        else:
            new_w = in_w
            new_h = int(new_w * im_ratio)

        
        resized = cv2.resize(img, (new_w, new_h))
        canvas = np.zeros((in_h, in_w, 3), dtype=np.uint8)
        canvas[:new_h, :new_w] = resized

        # Forward
        outs = self.forward(canvas)
        # Prepare lists
        scores_list, bboxes_list, kps_list = [], [], []

        # Split outputs
        # confidences: 0..fmc-1, bboxes: fmc..2*fmc-1, kps: 2*fmc.. if use_kps
        for idx, stride in enumerate(self._feat_stride_fpn):
            # Score
            sc = outs[idx].cpu().numpy().squeeze()
            # BBox preds
            bd = outs[idx + self.fmc].cpu().numpy()
            bd = bd * stride

            # KPS preds
            if self.use_kps:
                kp = outs[idx + self.fmc*2].cpu().numpy() * stride
                
            # Grid size
            height = canvas.shape[0] // stride
            width  = canvas.shape[1] // stride
            key = (height, width, stride)
           # Anchor centers *exactly* as in the ONNX wrapper*
            if key in self.center_cache:
                centers = self.center_cache[key]
            else:
                # 1) build a (height×width×2) array where [:,:,0]=x indices, [:,:,1]=y indices
                ac = np.stack(
                    np.mgrid[:height, :width][::-1],  # yields [x_grid, y_grid]
                    axis=-1
                ).astype(np.float32)
                # 2) scale by stride and flatten to (N,2)
                ac = (ac * stride).reshape(-1, 2)
                # 3) if using >1 anchor per location, replicate each center exactly
                if self._num_anchors > 1:
                    ac = np.stack([ac] * self._num_anchors, axis=1).reshape(-1, 2)
                centers = ac
                self.center_cache[key] = centers



            # Decode bboxes
            # distance2bbox: [x_center,y_center] + [l,t,r,b] -> [x1,y1,x2,y2]
            from insightface.model_zoo.retinaface import distance2bbox, distance2kps
            bbs = distance2bbox(centers, bd.reshape(-1, 4))
            # Filter by threshold
            pos = np.where(sc.ravel() >= self.det_thresh)[0]
            scores_list.append(sc.ravel()[pos])
            bboxes_list.append(bbs[pos])
            if self.use_kps:
                kpss = distance2kps(centers, kp.reshape(-1, kp.shape[-1]))
                kpss = kpss.reshape(-1, kp.shape[-1]//2, 2)
                kps_list.append(kpss[pos])

        # Stack results
        scores = np.vstack([s.reshape(-1,1) for s in scores_list])
        bboxes = np.vstack(bboxes_list)
        if self.use_kps:
            kpss = np.vstack(kps_list)
        else:
            kpss = None

        # Sort by score
        order = scores[:,0].argsort()[::-1]
        pre = np.hstack((bboxes, scores)).astype(np.float32)
        pre = pre[order]
        if kpss is not None:
            kpss = kpss[order]

        # Apply NMS
        keep = self.nms(pre)
        det = pre[keep]
        if kpss is not None:
            kpss = kpss[keep]

        # Clip and scale back to original image size
        scale = new_h / img.shape[0] if im_ratio>model_ratio else new_w / img.shape[1]
        det[:,:4] /= scale
        if kpss is not None:
            kpss /= scale

        # Optionally limit number of detections
        if max_num > 0 and len(det) > max_num:
            areas = (det[:,2]-det[:,0])*(det[:,3]-det[:,1])
            centers_img = np.array([(det[:,0]+det[:,2])/2, (det[:,1]+det[:,3])/2]).T
            img_center = np.array([img.shape[1]/2, img.shape[0]/2])
            dists = np.sum((centers_img - img_center)**2, axis=-1)
            metrics = areas - 2*dists if metric=='default' else areas
            idxs = np.argsort(metrics)[::-1][:max_num]
            det = det[idxs]
            if kpss is not None:
                kpss = kpss[idxs]

        return det, kpss

    
    def nms(self, dets):
        """Pure NumPy NMS."""
        x1, y1, x2, y2, scores = dets[:,0], dets[:,1], dets[:,2], dets[:,3], dets[:,4]
        areas = (x2 - x1 + 1) * (y2 - y1 + 1)
        order = scores.argsort()[::-1]
        keep = []
        while order.size > 0:
            i = order[0]
            keep.append(i)
            xx1 = np.maximum(x1[i], x1[order[1:]])
            yy1 = np.maximum(y1[i], y1[order[1:]])
            xx2 = np.minimum(x2[i], x2[order[1:]])
            yy2 = np.minimum(y2[i], y2[order[1:]])
            w = np.maximum(0.0, xx2 - xx1 + 1)
            h = np.maximum(0.0, yy2 - yy1 + 1)
            inter = w * h
            ovr = inter / (areas[i] + areas[order[1:]] - inter)
            inds = np.where(ovr <= self.nms_thresh)[0]
            order = order[inds + 1]
        return keep
    

    def predict(self, img: np.ndarray, bboxes=None) -> List[Tuple[float, float, float, float]]:
        """Wrapper around detect(), ignoring `bboxes` arg."""
        dets, kpss = self.detect(img)
        return dets.tolist(), kpss.tolist()


class AgeGenderModel(FaceModelBase):
    def __init__(self, model_path: str, det_size: Tuple[int, int] = (640, 640), face_size: Tuple[int, int] = (96, 96)):
        assert model_path is not None and os.path.exists(model_path), f"Model file not found: {model_path}"
        self.model_path = model_path
        # self.input_mean = 127.5
        # self.input_std = 128.0
        
        # important !!!
        self.input_mean = 0
        self.input_std = 1


        self.det_size = det_size
        self.face_size = face_size

        model = onnx.load(model_path)

        # Convert ONNX to Torch
        self.torch_model = convert(model)
        self.torch_model.eval()

    def prepare(self, *args, **kwargs):
        pass  # Nothing to prepare, or set `input_mean/std` dynamically here


    def preprocess_face(self, img: np.ndarray) -> torch.Tensor:
        img = cv2.resize(img, self.face_size)  # size = (W, H)

        # Convert BGR to RGB
        img = img[:, :, ::-1]  # OpenCV loads BGR; we want RGB

        # Convert to float32 and scale
        img = img.astype(np.float32)
        img = (img - self.input_mean) / self.input_std

        # HWC → CHW
        img = img.transpose(2, 0, 1)

        # Convert to tensor
        return torch.from_numpy(img)

    def predict(self, img: np.ndarray, bboxes: List[Tuple[float, float, float, float]]) -> List[Tuple[int, int]]:
        """
        Predict gender and age for each face in the given image.
        Args:
            img     : Original image (HWC, BGR or RGB)
            bboxes  : List of bounding boxes, each (x1, y1, x2, y2)
        Returns:
            List of (gender, age) tuples
        """
        results = []
        rotate = 0
        for bbox in bboxes:
            w, h = bbox[2] - bbox[0], bbox[3] - bbox[1]
            center = ((bbox[0] + bbox[2]) / 2, (bbox[1] + bbox[3]) / 2)
            scale = self.face_size[0] / (max(w, h) * 1.5)


            aligned_face, _ = transform(img, center, self.face_size[0], scale, rotate)

            if aligned_face is None or aligned_face.size == 0:
                results.append((None, None))
                continue


            tensor = self.preprocess_face(aligned_face).unsqueeze(0)
            with torch.no_grad():
                output = self.torch_model(tensor)[0].cpu().numpy()

            gender = int(np.argmax(output[:2]))
            age = int(np.round(output[2] * 100))
            results.append((gender, age))
            

        return results

    

# Example usage:
image = cv2.imread("data/images/000000495166.jpg")

actual = app.get(image)
print(f"Actual predictions:")
print(f"Gender: {actual[0].gender}, Age: {actual[0].age}")
print(f"Bbox: {actual[0].bbox}")

rft = RetinaFaceTorch(app.models["detection"].model_file)
rft.prepare(nms_thresh=0.4, det_thresh=0.5, input_size=(640,640))
dets, kpss = rft.predict(image)
print("Detections:", dets)


age_model = AgeGenderModel(app.models["genderage"].model_file)
age_model.prepare()
preds = age_model.predict(image, dets)
print(preds)






In [ ]:
import cv2
from PIL import Image
from IPython.display import display

# 1) Load your image
img_bgr = cv2.imread("data/images/000000495166.jpg")

# 2) Get prediction & bbox
actual = app.get(img_bgr)
x1, y1, x2, y2 = map(int, actual[0].bbox)
label = f"G: {actual[0].gender}, A: {actual[0].age}"

# 3) Draw the rectangle and label directly on the BGR image
img = img_bgr.copy()
cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
cv2.putText(
    img, label, (x1, y1 - 10),
    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2, lineType=cv2.LINE_AA
)

# 4) Convert to RGB for correct colors
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# 5) Wrap in a PIL image and display
pil_img = Image.fromarray(img_rgb)
display(pil_img)




In [ ]:
import cv2
from PIL import Image
from IPython.display import display

# 1) Load your image (BGR)
img_bgr = cv2.imread("data/images/000000495166.jpg")

# 2) Run your detector
actual = app.get(img_bgr)
x1, y1, x2, y2 = map(int, actual[0].bbox)  # bbox coords
label = f"G: {actual[0].gender}, A: {actual[0].age}"

# 3) Draw the bounding box + label on a copy
img_annot = img_bgr.copy()
cv2.rectangle(img_annot, (x1, y1), (x2, y2), (0, 255, 0), 2)
# cv2.putText(
#     img_annot, label, (x1, y1 - 10),
#     cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2, lineType=cv2.LINE_AA
# )

# 4) Convert to RGB and display full annotated image
img_annot_rgb = cv2.cvtColor(img_annot, cv2.COLOR_BGR2RGB)
display(Image.fromarray(img_annot_rgb))

# 5) Crop out the bbox region from the original (no labels)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
crop = img_rgb[y1:y2, x1:x2]

# 6) Display the cropped face only
display(Image.fromarray(crop))
